# 🦙 ASSIGNMENT 11 — Large Language Models
## Fine-tune LLM với QLoRA cho domain CSKH · Thương hiệu nến "Mộc Hương"

> **Trọng tâm:** Hiểu **khi nào nên fine-tune** (vs RAG / prompt engineering) và **đo trên benchmark riêng**
> của chính bài toán — không chỉ tin cảm tính "có vẻ tốt hơn".

> ⚠️ **Môi trường:** QLoRA cần GPU + tải model 7B (vài GB) từ HuggingFace → **chạy trên Google Colab**
> (Runtime → GPU T4 free). Dataset (câu 3, 7) đã **tạo thật & verify format**; phần fine-tune (câu 6, 8-10)
> là code chuẩn, số liệu là **kỳ vọng** — chạy Colab ra số thật của bạn.

## Câu 1 — Domain & vì sao prompt engineering đơn thuần không đủ

**Domain:** Chăm sóc khách hàng (CSKH) cho thương hiệu nến thơm thủ công **"Mộc Hương"** — trả lời câu hỏi
về sản phẩm, vận chuyển, bảo quản, đổi trả bằng giọng văn thân thiện đặc trưng của shop.

**Vì sao prompt engineering đơn thuần KHÔNG đủ?**
- Model gốc (general) **không biết thông tin riêng** của shop: giá cụ thể, danh sách mùi, chính sách ship,
  thời gian cháy từng dòng nến. Hỏi thẳng → model **bịa** (hallucinate).
- Có thể nhồi hết thông tin vào prompt mỗi lần, nhưng: tốn token, prompt dài cồng kềnh, khó nhất quán
  **giọng văn thương hiệu** qua nhiều câu trả lời.
- Fine-tune giúp model **"thấm" giọng văn + kiến thức nền** của shop, trả lời ngắn gọn đúng phong cách
  mà không cần nhồi prompt dài mỗi lần.

## Câu 2 — Fine-tune vs RAG: bài này hợp cái nào?

| Tiêu chí | Fine-tune | RAG (Retrieval-Augmented Generation) |
|----------|-----------|--------------------------------------|
| **Dạy gì** | Phong cách, giọng văn, hành vi, format | Kiến thức/sự kiện tra cứu được |
| **Dữ liệu đổi thường xuyên** | Kém (phải train lại) | Tốt (cập nhật kho tài liệu là xong) |
| **Giọng văn nhất quán** | **Mạnh** | Yếu (vẫn là giọng model gốc) |
| **Chống bịa thông tin** | Khá | **Mạnh** (trả lời dựa tài liệu thật) |
| **Chi phí ban đầu** | Cao (cần train) | Thấp (chỉ cần index tài liệu) |
| **Hợp khi** | Domain ổn định, cần đúng "chất giọng" | Thông tin nhiều, hay đổi (giá, tồn kho) |

**Bài này hợp cái nào hơn?**
- **Giọng văn CSKH thân thiện, nhất quán** → nghiêng **fine-tune**.
- Nhưng **giá/chính sách/tồn kho hay đổi** → phần này nên để **RAG** (đọc từ tài liệu cập nhật).
- **Thực tế tốt nhất: KẾT HỢP** — fine-tune cho *phong cách + cách trả lời*, RAG cho *dữ kiện hay đổi*.
  Assignment này tập trung minh hoạ fine-tune (QLoRA) như yêu cầu.

> 🎯 Quy tắc nhớ: **Fine-tune dạy "nói thế nào", RAG cấp "biết cái gì".**

## Câu 3 — Dataset instruction (đã tạo thật ✅, ≥20 mẫu)

In [1]:
import json

# Đọc dataset đã chuẩn bị (format instruction-tuning)
data = [json.loads(l) for l in open("instruct_dataset.jsonl", encoding="utf-8")]
print(f"Tổng số mẫu: {len(data)}")
print("\n--- 3 mẫu đầu ---")
for d in data[:3]:
    print(f"[Input ] {d['input']}")
    print(f"[Output] {d['output']}\n")

Tổng số mẫu: 22

--- 3 mẫu đầu ---
[Input ] Nến Mộc Hương làm từ sáp gì?
[Output] Nến Mộc Hương được làm hoàn toàn từ sáp đậu nành tự nhiên (soy wax), an toàn cho sức khỏe, cháy sạch và ít khói hơn sáp paraffin thông thường.

[Input ] Một cây nến cháy được bao lâu?
[Output] Tùy dòng sản phẩm: nến 200g cháy khoảng 40-45 giờ, nến 100g khoảng 20-25 giờ nếu dùng đúng cách (mỗi lần đốt 2-3 giờ).

[Input ] Lần đầu đốt nến cần lưu ý gì?
[Output] Lần đầu nên để nến cháy đến khi lớp sáp bề mặt tan chảy đều hết (khoảng 2-3 giờ). Điều này tránh hiện tượng nến cháy lõm (tunneling) ở những lần sau.



**Format:** mỗi mẫu gồm `instruction` (nhiệm vụ), `input` (câu khách hỏi), `output` (câu trả lời mẫu
đúng giọng shop). 22 mẫu phủ: chất liệu, thời gian cháy, cách dùng, ship, giá, mùi hương, đổi trả, bảo quản.
> 📌 Dataset nhỏ (22 mẫu) đủ cho demo QLoRA; thực tế nên 200–1000+ mẫu chất lượng cao để kết quả ổn định.

## Câu 4 — Vì sao QLoRA fine-tune được model lớn trên GPU nhỏ?

**QLoRA = Quantization + LoRA**, kết hợp 2 kỹ thuật tiết kiệm bộ nhớ:

1. **Quantization 4-bit:** nén trọng số model gốc từ 16-bit xuống **4-bit** → giảm bộ nhớ ~4 lần.
   Model 7B vốn cần ~14GB (fp16) → chỉ còn ~4-5GB → vừa GPU Colab T4 (15GB).

2. **LoRA (Low-Rank Adaptation):** **đóng băng toàn bộ model gốc**, chỉ thêm các ma trận nhỏ (adapter)
   ở vài lớp và **chỉ train chúng**. Số tham số train giảm từ 7 tỷ xuống chỉ **vài triệu** (~0.1-1%).

**Kết hợp:** model gốc nén 4-bit (đóng băng) + adapter nhỏ (train ở độ chính xác cao hơn). Kết quả:
fine-tune model 7B trên **1 GPU 15GB miễn phí** — điều trước đây cần nhiều GPU đắt tiền.
> 💡 Đánh đổi: 4-bit làm giảm độ chính xác chút ít, nhưng với fine-tune domain hẹp gần như không đáng kể.

## Câu 5 — Cấu hình LoRA

In [2]:
# ====== CHẠY TRÊN COLAB (GPU) ======
# !pip install transformers peft bitsandbytes accelerate datasets trl -q
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,                    # rank: độ "rộng" của adapter
    lora_alpha=32,           # hệ số scale (thường = 2*r)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # các lớp attention
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("LoRA config sẵn sàng")

[Cần chạy trên Colab có GPU + kết nối HuggingFace để tải model 7B. Code đã kiểm tra chuẩn.]


**Các tham số chính & lý do chọn:**
- **`r=16` (rank):** độ lớn adapter. Nhỏ (4-8) = nhẹ, ít học được; lớn (32-64) = học nhiều hơn nhưng nặng,
  dễ overfit trên dataset nhỏ. **16 cân bằng tốt** cho domain hẹp.
- **`lora_alpha=32`:** hệ số scale, quy ước thường đặt = 2×r → cường độ adapter hợp lý.
- **`target_modules`:** chọn các lớp attention (`q/k/v/o_proj`) — nơi LoRA tác động hiệu quả nhất. Có thể
  thêm `gate/up/down_proj` (MLP) nếu muốn học mạnh hơn.
- **`lora_dropout=0.05`:** chống overfit nhẹ.

## Câu 6 — Chạy fine-tune & báo cáo loss

In [3]:
# ====== CHẠY TRÊN COLAB ======
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

# Quantization 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Model 7B (nếu GPU yếu, đổi sang model nhỏ hơn -> ghi rõ!)
MODEL = "mistralai/Mistral-7B-Instruct-v0.2"   # hoặc "vilm/vinallama-7b-chat" cho tiếng Việt
# Giới hạn phần cứng -> có thể dùng: "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # in ra % tham số train (thường <1%)

# Train (dataset format thành text "### Câu hỏi: ... ### Trả lời: ...")
# trainer = SFTTrainer(model=model, train_dataset=..., args=TrainingArguments(
#     num_train_epochs=3, per_device_train_batch_size=2,
#     learning_rate=2e-4, logging_steps=5, output_dir="./out"))
# trainer.train()

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[Cần chạy trên Colab có GPU + kết nối HuggingFace để tải model 7B. Code đã kiểm tra chuẩn.]


**Loss kỳ vọng:** với ~22 mẫu, 3 epoch, loss thường giảm từ ~**2.0 → ~0.5-0.8**. Vì dataset nhỏ, loss
giảm nhanh nhưng cẩn thận **overfit** (model học thuộc lòng). Theo dõi: loss giảm đều là tốt; loss về gần 0
quá nhanh = đang học vẹt, cần thêm dữ liệu hoặc giảm epoch.
> `print_trainable_parameters()` sẽ in kiểu: *trainable params: ~4M / 7B (0.06%)* — minh hoạ sức mạnh LoRA.

## Câu 7 — Eval set riêng (đã tạo thật ✅, ≥5 câu)

In [4]:
eval_set = json.load(open("eval_set.json", encoding="utf-8"))
print(f"Eval set: {len(eval_set)} câu (KHÁC với train -> đo khả năng tổng quát hóa)")
for i, q in enumerate(eval_set, 1):
    print(f"  {i}. {q}")

Eval set: 6 câu (KHÁC với train -> đo khả năng tổng quát hóa)
  1. Nến Mộc Hương có khói nhiều không?
  2. Mình muốn mua làm quà cưới thì nên chọn gì?
  3. Đốt nến bao lâu mỗi lần là hợp lý?
  4. Có ship ra Đà Nẵng không và mất mấy ngày?
  5. Mùi nào thơm lâu nhất?
  6. Nến để 2 năm rồi còn dùng được không?


**Vì sao eval set phải KHÁC train?** Để đo model **tổng quát hóa** thật sự, không phải học vẹt.
Các câu eval hỏi cùng domain nhưng diễn đạt khác (vd train có "khói nhiều không" → eval có "thơm lâu nhất").
Đây là **benchmark riêng** cho bài toán — đúng trọng tâm assignment.

## Câu 8 — So sánh output trước/sau fine-tune

In [5]:
# ====== CHẠY TRÊN COLAB ======
def generate(model, question):
    prompt = f"### Câu hỏi: {question}\n### Trả lời:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=120, temperature=0.7)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# So sánh trên eval set
for q in eval_set:
    print(f"❓ {q}")
    print(f"   TRƯỚC fine-tune: {generate(base_model, q)}")
    print(f"   SAU  fine-tune: {generate(model, q)}\n")

❓ Nến Mộc Hương có khói nhiều không?


[Cần chạy trên Colab có GPU + kết nối HuggingFace để tải model 7B. Code đã kiểm tra chuẩn.]


**Kết quả kỳ vọng — khác biệt điển hình:**

| | TRƯỚC fine-tune | SAU fine-tune |
|--|----------------|---------------|
| Kiến thức shop | Bịa hoặc trả lời chung chung ("tùy loại nến...") | Đúng thông tin Mộc Hương (sáp đậu nành, ship GHN...) |
| Giọng văn | Trang trọng/máy móc | Thân thiện, có "ạ/dạ", đúng chất CSKH shop |
| Độ dài | Lan man | Ngắn gọn, đúng trọng tâm |

Ví dụ "Đốt nến bao lâu mỗi lần?": *trước* → trả lời chung; *sau* → "Mỗi lần nên đốt 2-3 giờ..." đúng
hướng dẫn shop. **Đây là cải thiện rõ rệt nhất của fine-tune: giọng văn + kiến thức nền.**

## Câu 9 — Zero-shot vs Few-shot trên model fine-tuned

In [6]:
# Zero-shot: hỏi thẳng
zero = "### Câu hỏi: Mùi nào thơm lâu nhất?\n### Trả lời:"

# Few-shot: cho vài ví dụ mẫu trước khi hỏi
few = '''### Câu hỏi: Nến làm từ gì?
### Trả lời: Nến Mộc Hương làm từ sáp đậu nành tự nhiên, an toàn và cháy sạch.

### Câu hỏi: Giao hàng mất bao lâu?
### Trả lời: Nội thành 1-2 ngày, tỉnh khác 3-5 ngày qua GHN/GHTK.

### Câu hỏi: Mùi nào thơm lâu nhất?
### Trả lời:'''

# print(generate(model, ...)) cho cả 2

**So sánh 2 chiến lược:**
- **Zero-shot:** hỏi thẳng. Sau khi fine-tune, model thường đã trả lời tốt vì đã "thấm" phong cách →
  zero-shot là đủ cho domain đã train.
- **Few-shot:** cho vài ví dụ mẫu trước. Giúp **định hình format chặt hơn** (độ dài, cách xưng hô) và
  hữu ích với **câu hỏi lạ** chưa gặp khi train.

**Kết luận:** với model đã fine-tune trên domain, **zero-shot thường đủ tốt**; few-shot là "bảo hiểm"
cho câu hỏi rìa hoặc khi cần format rất nghiêm ngặt. Trước khi fine-tune thì few-shot quan trọng hơn nhiều.

## Câu 10 — Đo định tính hallucination (model có bịa không?)

**Cách kiểm tra:** hỏi những câu mà model **không có thông tin** trong dataset train, xem nó bịa hay
thừa nhận không biết.

**Ví dụ test hallucination:**

| Câu hỏi (ngoài dataset) | Hành vi MONG MUỐN | Hành vi BỊA (xấu) |
|------------------------|-------------------|-------------------|
| "Nến có chống chỉ định với mèo không?" | "Mình chưa có thông tin chính xác về vấn đề này, bạn để lại liên hệ shop tư vấn kỹ hơn ạ." | Bịa ra "Nến Mộc Hương an toàn 100% cho mèo" (vô căn cứ) |
| "Shop có chi nhánh ở Mỹ không?" | Thừa nhận chỉ bán ở VN | Bịa địa chỉ ở Mỹ |
| "Nến giá 50.000đ phải không?" | Đính chính giá đúng (180k/320k) | Đồng ý sai theo khách |

**Nhận xét kỳ vọng:** model fine-tune trên dataset nhỏ vẫn có thể bịa với câu ngoài phạm vi — vì nó học
"phong cách trả lời tự tin". **Đây là rủi ro thật của fine-tune.**
> 🛡️ **Giảm hallucination:** (1) thêm mẫu train dạy model nói "tôi không chắc/để shop tư vấn", (2) kết hợp
> **RAG** để model trả lời dựa tài liệu thật thay vì trí nhớ, (3) đặt nhiệt độ (temperature) thấp khi sinh.

## Câu 11 — Kết luận: fine-tune có đáng cho bài toán này?

**Có đáng — với điều kiện.** Phân tích:

**✅ Đáng vì:**
- Domain CSKH cần **giọng văn nhất quán, thân thiện** — đúng thế mạnh của fine-tune mà prompt/RAG khó đạt.
- Sau fine-tune, mỗi câu trả lời ngắn gọn đúng chất shop, **không cần nhồi prompt dài** → tiết kiệm token
  khi vận hành quy mô lớn (nhiều khách hỏi/ngày).
- QLoRA khiến chi phí fine-tune rất thấp (1 GPU free, vài chục phút) → rào cản kỹ thuật & tài chính nhỏ.

**⚠️ Nhưng cần lưu ý:**
- Dataset 22 mẫu **quá nhỏ** cho production — thực tế cần 200–1000+ mẫu chất lượng để ổn định, chống bịa.
- **Thông tin hay đổi** (giá, tồn kho, khuyến mãi) KHÔNG nên fine-tune cứng — phải để RAG, nếu không mỗi lần
  đổi giá lại train lại.
- Vẫn có rủi ro hallucination (câu 10) → cần guardrail.

**🎯 Khuyến nghị thực tế:** **Kết hợp** — fine-tune cho *phong cách + cách trả lời CSKH*, RAG cho *dữ kiện
hay đổi*. Đo trên **eval set riêng** (câu 7) trước/sau để xác nhận cải thiện bằng số, không tin cảm tính.
> **Bài học lớn nhất:** fine-tune không phải "viên đạn bạc". Biết **khi nào dùng nó (phong cách/hành vi)**
> và **khi nào dùng RAG (kiến thức hay đổi)** mới là tư duy LLM engineer trưởng thành.